<a href="https://colab.research.google.com/github/LeonardoSuperno/RL_methods/blob/main/DPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate peft trl bitsandbytes

from google.colab import drive
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/dpo_llama_model"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Mounted at /content/drive


Load Llama 3.2 1B

In [ ]:
from huggingface_hub import login
login()

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"



config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("allenai/Dolci-Think-DPO-7B", split="train")

# subset
dataset = dataset.shuffle(seed=42).select(range(5500))

# 5k train, 1k eval
train_dataset = dataset.select(range(5000))
eval_dataset  = dataset.select(range(5000, 5500))


def format_chat(example):
    prompt = example["prompt"]
    chosen = example["chosen"]
    rejected = example["rejected"]

    # costruzione messaggi chat
    messages = [
        {"role": "user", "content": prompt}
    ]

    prompt_formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    example["prompt"] = prompt_formatted
    example["chosen"] = chosen
    example["rejected"] = rejected

    return example

train_dataset = train_dataset.map(format_chat)
eval_dataset  = eval_dataset.map(format_chat)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/199M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/200M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/199M [00:00<?, ?B/s]

...

data/train-00005-of-00007.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

data/train-00006-of-00007.parquet:   0%|          | 0.00/199M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Fine-tuning

In [ ]:
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [ ]:
from trl import DPOTrainer, DPOConfig

training_args = DPOConfig(
    output_dir="./dpo-llama-1b",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="no",
    learning_rate=5e-5,
    max_length=512,
    max_prompt_length=256,
    fp16=True,
)

ref_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modello salvato in {output_dir}")

/tmp/ipython-input-782954315.py:3: FutureWarning: `max_prompt_length` is deprecated and will be removed in version 0.29.0. We recommend filtering out overlong prompts from your dataset before passing it to the trainer instead of using this parameter.
  training_args = DPOConfig(


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Extracting prompt in train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
20,0.557882
40,0.290127
60,0.236805
80,0.197565
100,0.175191
120,0.145886
140,0.141218
160,0.136257
180,0.165568
200,0.144070


Modello salvato in /content/drive/MyDrive/dpo_llama_model


Eval

In [ ]:
# RUN ONLY TO LOAD THE FINE-TUNED MODEL

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the fine-tuned tokenizer
tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Load the fine-tuned model
model = AutoModelForCausalLM.from_pretrained(
    output_dir,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Fine-tuned model and tokenizer loaded successfully from {output_dir}")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/64 [00:00<?, ?it/s]

Fine-tuned model and tokenizer loaded successfully from /content/drive/MyDrive/dpo_llama_model


In [ ]:
reference_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
import torch
import torch.nn.functional as F

def compute_logprob_batch(
    model,
    tokenizer,
    prompts,
    responses,
    max_length=1024,
    average_log_prob=True,   # se False usa sum invece di mean
):
    """
    Computes log p(response | prompt) for a batch.

    Returns:
        List[float]: log-probability (mean or sum) for each (prompt, response)
    """

    device = model.device
    pad_token_id = tokenizer.pad_token_id

    batch_input_ids = []
    batch_attention_mask = []
    batch_prompt_lengths = []
    batch_response_lengths = []

    # -------------------------
    # Tokenization
    # -------------------------
    for prompt, response in zip(prompts, responses):

        prompt_ids = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            add_special_tokens=False,
        )["input_ids"].squeeze(0)

        response_ids = tokenizer(
            response,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            add_special_tokens=False,
        )["input_ids"].squeeze(0)

        combined_ids = torch.cat([prompt_ids, response_ids], dim=0)

        batch_input_ids.append(combined_ids)
        batch_attention_mask.append(torch.ones_like(combined_ids))

        batch_prompt_lengths.append(prompt_ids.shape[0])
        batch_response_lengths.append(response_ids.shape[0])

    # -------------------------
    # Padding
    # -------------------------
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        batch_input_ids,
        batch_first=True,
        padding_value=pad_token_id,
    ).to(device)

    padded_attention_mask = torch.nn.utils.rnn.pad_sequence(
        batch_attention_mask,
        batch_first=True,
        padding_value=0,
    ).to(device)

    # -------------------------
    # Forward pass
    # -------------------------
    model.eval()
    with torch.no_grad():

        outputs = model(
            input_ids=padded_input_ids,
            attention_mask=padded_attention_mask,
        )

        logits = outputs.logits  # (B, T, V)

        # Causal shift
        shift_logits = logits[:, :-1, :]
        shift_labels = padded_input_ids[:, 1:]

        log_probs = F.log_softmax(shift_logits, dim=-1)

        # Gather log-probs of target tokens
        token_log_probs = torch.gather(
            log_probs,
            dim=-1,
            index=shift_labels.unsqueeze(-1),
        ).squeeze(-1)

        # Mask padding tokens
        loss_mask = shift_labels != pad_token_id
        token_log_probs = token_log_probs * loss_mask

        results = []

        # -------------------------
        # Extract response scores
        # -------------------------
        for i in range(len(prompts)):

            prompt_len = batch_prompt_lengths[i]
            response_len = batch_response_lengths[i]

            if response_len == 0:
                results.append(0.0)
                continue

            # Start index of response log-probs
            start = prompt_len - 1
            end = start + response_len

            response_token_log_probs = token_log_probs[i, start:end]

            total_logprob = response_token_log_probs.sum()

            if average_log_prob:
                score = total_logprob / response_len
            else:
                score = total_logprob

            results.append(score.item())

    return results

In [ ]:
from tqdm import tqdm
import torch

def pairwise_accuracy_batched(
    policy_model,
    reference_model,
    tokenizer,
    dataset,
    batch_size=4,
    max_length=1024,
    average_log_prob=True,
):

    correct = 0
    total = len(dataset)

    policy_model.eval()

    reference_model.eval()
    for param in reference_model.parameters():
      param.requires_grad = False

    with torch.no_grad():

        for i in tqdm(range(0, total, batch_size), desc="Elaborazione batch"):

            batch = dataset[i:i + batch_size]

            prompts = batch["prompt"]

            chosen_responses = [
                example[-1]["content"] for example in batch["chosen"]
            ]

            rejected_responses = [
                example[-1]["content"] for example in batch["rejected"]
            ]

            # -------------------------
            # Policy log-probs
            # -------------------------
            policy_chosen = compute_logprob_batch(
                policy_model,
                tokenizer,
                prompts,
                chosen_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )

            policy_rejected = compute_logprob_batch(
                policy_model,
                tokenizer,
                prompts,
                rejected_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )

            # -------------------------
            # Reference log-probs
            # -------------------------
            ref_chosen = compute_logprob_batch(
                reference_model,
                tokenizer,
                prompts,
                chosen_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )

            ref_rejected = compute_logprob_batch(
                reference_model,
                tokenizer,
                prompts,
                rejected_responses,
                max_length=max_length,
                average_log_prob=average_log_prob,
            )

            # -------------------------
            # DPO comparison
            # -------------------------
            for j in range(len(prompts)):

                chosen_score = policy_chosen[j] - ref_chosen[j]
                rejected_score = policy_rejected[j] - ref_rejected[j]

                if chosen_score > rejected_score:
                    correct += 1

    return correct / total

acc = pairwise_accuracy_batched(
    policy_model=model,
    reference_model=reference_model,
    tokenizer=tokenizer,
    dataset=eval_dataset,
    batch_size=4,
)

print("\nPairwise accuracy (DPO):", acc)

Elaborazione batch: 100%|██████████| 125/125 [09:03<00:00,  4.34s/it]


Pairwise accuracy (DPO): 0.972
